# Topic 1: Foundations — Why LLMOps?

**LLMOps Part 1 · Session 1 · Hands-On Lab**

---

### What This Notebook Covers

This notebook is a **learner-friendly, hands-on companion** to Topic 1 (Slides 1–7). It demonstrates — through runnable code — why LLM systems need a fundamentally different operational model than traditional software.

We work through **five progressive demos**, each tied to a core LLMOps concept:

| Demo | Concept | Ops Takeaway |
|---|---|---|
| 1 | Deterministic vs Probabilistic behaviour | You need **behaviour tests** (distribution/score), not exact-match assertions |
| 2 | Prompt drift regression test | Prompts must be **versioned + evaluated** like code |
| 3 | Hallucination vs grounded answers (RAG) | **Guardrails + grounding** reduce plausible-wrong outputs |
| 4 | Agent tool failure + retries | Agents are **distributed systems**; tool reliability needs engineering |
| 5 | Cost & latency explosion in multi-step chains | Cost + latency are **first-class SLOs** |

> **Goal:** After completing this notebook, you will understand *why* LLMOps exists — not as abstract theory, but because you will have seen each failure mode with your own eyes.

---

### Prerequisites

- An **OpenAI API key** (GPT-3.5-turbo / GPT-4o-mini)
- Runs on **Google Colab** or any Jupyter environment with internet access
- No prior LLMOps experience required

---
## 0. Setup and Dependencies

Install required packages and configure your API key. This cell only needs to run once per session. This Collab expects OPENAI_API_KEY to be made available in collab secrets

In [ ]:
!pip install -qU langchain-openai tiktoken numpy==2.1 pandas==2.2.3

In [ ]:
import os #, getpass, warnings
# warnings.filterwarnings("ignore")
from google.colab import userdata

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_APIKEY')

print("API key configured.")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import time, json, random, hashlib
import numpy as np
import pandas as pd

MODEL = "gpt-4o-mini"

def make_llm(temperature=0.7, max_tokens=300):
    return ChatOpenAI(model=MODEL, temperature=temperature, max_tokens=max_tokens)

def chat(prompt, system="You are a helpful assistant.", temperature=0.7, max_tokens=300):
    """Thin wrapper around LangChain ChatOpenAI that returns (AIMessage, elapsed_seconds)."""
    llm = make_llm(temperature=temperature, max_tokens=max_tokens)
    messages = [SystemMessage(content=system), HumanMessage(content=prompt)]
    start = time.time()
    resp = llm.invoke(messages)
    elapsed = time.time() - start
    return resp, elapsed

print(f"Using model: {MODEL} (via LangChain ChatOpenAI)")
print("Helper function `chat()` ready.")

---
## Demo 1: Deterministic vs Probabilistic Behaviour

### The Concept

Traditional software is **deterministic** — the same input always produces the same output.

LLMs are **probabilistic** — the same prompt can produce different outputs on each call.

### What We'll Do

Call the LLM **5 times** with the **exact same prompt** and observe how the outputs vary.

### Ops Takeaway

> You need **behaviour tests** (distribution/score-based), not exact-match assertions.

In [ ]:
PROMPT = "In one sentence, explain why monitoring is important for production AI systems."

print(f"Prompt: \"{PROMPT}\"")
print(f"Calling {MODEL} 5 times with temperature=0.7...\n")
print("=" * 80)

responses = []
for i in range(5):
    resp, elapsed = chat(PROMPT, temperature=0.7)
    text = resp.content.strip()
    responses.append(text)
    print(f"\nCall {i+1} ({elapsed:.2f}s):")
    print(f"  {text}")

print("\n" + "=" * 80)

### Measuring Variance

Let's quantify *how different* these responses are. We'll use a simple character-level hash to check for exact matches, and then measure average pairwise similarity.

In [ ]:
print(f"All responses:  {responses}")


hashes = [hashlib.md5(r.encode()).hexdigest() for r in responses]
print(f"All hashes:     {hashes}")

unique_hashes = len(set(hashes))
print(f"Unique responses: {unique_hashes} / {len(responses)}")

if unique_hashes == len(responses):
    print("\n>>> Every response is different — same input, different output.")
    print(">>> This is why exact-match testing BREAKS for LLM systems.")
else:
    print(f"\n>>> {len(responses) - unique_hashes} duplicates found.")
    print(">>> Even with some matches, variance is the norm, not the exception.")

SequenceMatcher is a class in Python's standard difflib library used for comparing pairs of sequences (strings, lists, etc.) to find similarities and differences

The basic idea behind SequenceMatcher() is to find the longest contiguous matching subsequence (LCS) that contains no “junk” elements (Junk would be common english filler words)

In [ ]:
from difflib import SequenceMatcher

similarities = []
for i in range(len(responses)):
    for j in range(i + 1, len(responses)):
        sim = SequenceMatcher(None, responses[i], responses[j]).ratio()
        similarities.append(sim)
print(f"All similarities: {similarities}")

avg_sim = np.mean(similarities)
min_sim = np.min(similarities)
max_sim = np.max(similarities)

print(f"Pairwise text similarity (SequenceMatcher):")
print(f"  Average: {avg_sim:.2%}")
print(f"  Min:     {min_sim:.2%}")
print(f"  Max:     {max_sim:.2%}")
print(f"\n>>> Responses are similar in MEANING but different in FORM.")
print(f">>> Traditional unit tests (assertEqual) would fail on every run.")

### What about temperature=0?

Even at temperature=0, outputs are **not guaranteed** to be identical across calls (API-level sampling, batching, and model updates can still introduce variation). Let's verify.

In [ ]:
print("Calling 5 times with temperature=0...\n")
det_responses = []
for i in range(5):
    resp, elapsed = chat(PROMPT, temperature=0)
    det_responses.append(resp.content.strip())

det_hashes = [hashlib.md5(r.encode()).hexdigest() for r in det_responses]
det_unique = len(set(det_hashes))
print(f"Unique responses at temp=0: {det_unique} / {len(det_responses)}")

if det_unique == 1:
    print(">>> All identical this time — but this is NOT guaranteed across API versions.")
else:
    print(">>> Even at temp=0, we got variation! This proves the point.")

print("\n>>> LESSON: Deterministic testing strategies do not work for LLM outputs.")
print(">>> You need scoring-based and distribution-based evaluation.")

---
## Demo 2: Prompt Drift Regression Test

### The Concept

A small change to a prompt can **silently degrade** output quality. Without versioning and evaluation, you won't catch it until users complain.

### What We'll Do

1. Define a **v1 prompt** (good) and a **v2 prompt** (slightly modified)
2. Run both against a **fixed evaluation set** of questions
3. Score the outputs with a simple LLM-as-judge scorer
4. Compare scores to detect the regression

### Ops Takeaway

> Prompts must be **versioned and evaluated** like code. Every change needs a regression test.

In [ ]:
PROMPT_V1 = """You are a precise technical assistant. Answer the question concisely
and accurately. If you are unsure, say "I don't know" rather than guessing.

Question: {question}"""

PROMPT_V2 = """Answer the following question. Be creative and provide interesting details.

Question: {question}"""

print("Prompt v1 (production — precise, safe):")
print(PROMPT_V1[:100] + "...")
print("\nPrompt v2 (proposed change — creative, less constrained):")
print(PROMPT_V2[:100] + "...")

In [ ]:
EVAL_SET = [
    {
        "question": "What is the capital of France?",
        "expected": "Paris",
    },
    {
        "question": "What does the acronym 'LLM' stand for in AI?",
        "expected": "Large Language Model",
    },
    {
        "question": "What is the time complexity of binary search?",
        "expected": "O(log n)",
    }
]

print(f"Evaluation set: {len(EVAL_SET)} questions")
for i, item in enumerate(EVAL_SET):
    print(f"  {i+1}. {item['question']}  →  expected: {item['expected']}")

In [ ]:
JUDGE_PROMPT = """You are an evaluation judge. Given a question, expected answer, and actual answer,
score the actual answer on a scale of 1-5:

5 = Perfect: correct, concise, no extra fluff
4 = Correct but slightly verbose or has minor extras
3 = Partially correct or significantly verbose
2 = Mostly incorrect or very off-topic additions
1 = Wrong or hallucinated

Question: {question}
Expected: {expected}
Actual: {actual}

Respond with ONLY a JSON object: {{"score": <int>, "reason": "<brief reason>"}}"""


def score_answer(question, expected, actual):
    """Use an LLM-as-judge to score an answer from 1-5."""
    prompt = JUDGE_PROMPT.format(question=question, expected=expected, actual=actual)
    resp, elapsed = chat(prompt, system="You are an evaluation judge.", temperature=0, max_tokens=100)
    text = resp.content.strip()
    try:
        result = json.loads(text)
        return result["score"], result["reason"]
    except (json.JSONDecodeError, KeyError):
        return None, f"Parse error: {text[:100]}"


print("Judge scorer ready. Let's run both prompt versions against the eval set.")

In [ ]:
results = []

for item in EVAL_SET:
    q = item["question"]
    expected = item["expected"]

    # Run v1
    resp_v1, _ = chat(PROMPT_V1.format(question=q), temperature=0)
    answer_v1 = resp_v1.content.strip()
    score_v1, reason_v1 = score_answer(q, expected, answer_v1)

    # Run v2
    resp_v2, _ = chat(PROMPT_V2.format(question=q), temperature=0)
    answer_v2 = resp_v2.content.strip()
    score_v2, reason_v2 = score_answer(q, expected, answer_v2)

    results.append({
        "question": q,
        "v1_answer": answer_v1[:80],
        "v1_score": score_v1,
        "v2_answer": answer_v2[:80],
        "v2_score": score_v2,
    })

print(f"results -->{results}")

df = pd.DataFrame(results)
print("\n" + "=" * 80)
print("PROMPT DRIFT REGRESSION RESULTS")
print("=" * 80)
df

In [ ]:
avg_v1 = df["v1_score"].mean()
avg_v2 = df["v2_score"].mean()
delta = avg_v2 - avg_v1

print(f"Average score — v1 (production): {avg_v1:.1f} / 5.0")
print(f"Average score — v2 (proposed):   {avg_v2:.1f} / 5.0")
print(f"Delta:                           {delta:+.1f}")
print()

if delta < -0.5:
    print("REGRESSION DETECTED: v2 scores lower than v1.")
    print(">>> This prompt change should be BLOCKED from production.")
elif delta > 0.5:
    print("IMPROVEMENT: v2 scores higher than v1.")
    print(">>> This prompt change can proceed to canary deployment.")
else:
    print("NO SIGNIFICANT CHANGE: scores are within tolerance.")
    print(">>> More eval data needed before making a decision.")

print("\n>>> LESSON: Without this eval gate, v2 would have gone to production silently.")
print(">>> Prompts MUST be versioned and regression-tested before deployment.")

---
## Demo 3: Hallucination vs Grounded Answers (RAG Failure Mode)

### The Concept

LLMs can produce **confident but incorrect** answers — hallucinations. Retrieval-Augmented Generation (RAG) grounds the model in real documents, reducing this risk.

### What We'll Do

1. Ask the LLM questions **only answerable from specific documents**
2. Compare: **no retrieval** (hallucination risk) vs **with retrieval** (grounded + citations)
3. Score both for factual accuracy

### Ops Takeaway

> **Guardrails + grounding** reduce plausible-wrong outputs. RAG is an operational pattern, not just a feature.

In [ ]:
KNOWLEDGE_BASE = {
    "doc_1": {
        "title": "Internal AI Policy",
        "content": (
            "InterviewKickstart's AI policy requires all LLM-powered features to pass a "
            "three-stage evaluation gate before production deployment. Stage 1 is automated "
            "scoring on a held-out test set. Stage 2 is human review of 50 random samples. "
            "Stage 3 is a 48-hour canary deployment with live traffic monitoring. The policy "
            "was last updated on January 15, 2026. Maximum allowed hallucination rate is 2%."
        ),
    },
    "doc_2": {
        "title": "LLM Cost Report Q4 2025",
        "content": (
            "In Q4 2025, the total LLM API spend was $47,250 across all products. The travel "
            "concierge agent accounted for 62% of total spend ($29,295). Average cost per "
            "request was $0.023. The top cost driver was the multi-step planning chain which "
            "averaged 4.2 LLM calls per user query. Caching reduced costs by 38%."
        ),
    },
    "doc_3": {
        "title": "Model Performance Dashboard - Feb 2026",
        "content": (
            "The primary production model is gpt-4o-mini. P95 latency is 1.8 seconds. "
            "Average response quality score (LLM-as-judge) is 4.2/5.0. Drift detection "
            "flagged a 7% shift in input distribution on Feb 3, 2026, traced to a new "
            "user segment from the marketing campaign. The model was retrained on Feb 10."
        ),
    },
}

print(f"Knowledge base loaded: {len(KNOWLEDGE_BASE)} documents")
for doc_id, doc in KNOWLEDGE_BASE.items():
    print(f"  {doc_id}: {doc['title']}")

In [ ]:
QUESTIONS = [
    "What is the maximum allowed hallucination rate in the AI policy?",
    "How much did the travel concierge agent cost in Q4 2025?",
    "When was the input distribution drift detected, and what caused it?",
    "How many evaluation stages are required before production deployment?",
    "What was the average cost per LLM request?",
]

print("Questions (only answerable from the knowledge base):")
for i, q in enumerate(QUESTIONS):
    print(f"  {i+1}. {q}")

In [ ]:
def retrieve(question, kb=KNOWLEDGE_BASE):
    """Simple keyword retrieval: return documents that share words with the question."""
    q_words = set(question.lower().split())
    scored = []
    for doc_id, doc in kb.items():
        doc_words = set(doc["content"].lower().split())
        overlap = len(q_words & doc_words)
        scored.append((overlap, doc_id, doc))
    scored.sort(reverse=True)
    return scored[:2]  # top 2 documents


RAG_SYSTEM = """You are a precise assistant. Answer ONLY using the provided CONTEXT documents.
If the answer is not in the context, say "I don't have that information."
Always cite which document you used (e.g., [Doc: Internal AI Policy]).

CONTEXT:
{context}"""


print("Retriever and RAG system prompt ready.")

In [ ]:
rag_results = []

for q in QUESTIONS:
    # Without retrieval (hallucination risk)
    resp_no_rag, _ = chat(q, system="You are a helpful assistant.", temperature=0)
    answer_no_rag = resp_no_rag.content.strip()

    # With retrieval (grounded)
    retrieved = retrieve(q)
    context = "\n\n".join(
        f"[Doc: {doc['title']}]\n{doc['content']}" for _, _, doc in retrieved
    )
    system_with_context = RAG_SYSTEM.format(context=context)
    resp_rag, _ = chat(q, system=system_with_context, temperature=0)
    answer_rag = resp_rag.content.strip()

    rag_results.append({
        "question": q,
        "no_retrieval": answer_no_rag[:120],
        "with_retrieval": answer_rag[:120],
        "docs_used": ", ".join(doc["title"] for _, _, doc in retrieved),
    })

rag_df = pd.DataFrame(rag_results)
print("\n" + "=" * 80)
print("HALLUCINATION vs GROUNDED ANSWERS")
print("=" * 80)
for _, row in rag_df.iterrows():
    print(f"\nQ: {row['question']}")
    print(f"  WITHOUT RAG: {row['no_retrieval']}")
    print(f"  WITH RAG:    {row['with_retrieval']}")
    print(f"  Docs used:   {row['docs_used']}")

### What Did We See?

- **Without retrieval:** The LLM either made up numbers, gave generic answers, or hallucinated plausible-sounding details that are completely wrong.
- **With retrieval:** The answers are grounded in real documents and include citations.

> **Ops Takeaway:** Hallucination is not a bug you fix once — it is a **continuous operational risk**. RAG with source attribution is a guardrail, not just a feature. You must monitor hallucination rates as a production metric.

---
## Demo 4: Agent Tool Failure + Retries + Timeouts

### The Concept

LLM agents call external tools (APIs, databases, search). These tools can **fail, timeout, or return malformed data**. A naive agent crashes. A robust agent handles failures gracefully.

### What We'll Do

1. Simulate a tool that **randomly times out** or **returns malformed JSON**
2. Show a **naive agent** that fails on first error
3. Show a **robust agent** with retry logic, backoff, and schema validation

### Ops Takeaway

> Agents are **distributed systems**. Tool reliability needs engineering: retries, timeouts, circuit breakers, and schema validation.

In [12]:
def unreliable_tool(query: str) -> dict:
    """Simulates a tool that fails ~40% of the time."""
    roll = random.random()

    if roll < 0.2:
        # Simulate timeout
        time.sleep(3)
        raise TimeoutError(f"Tool timed out after 3s for query: {query}")

    elif roll < 0.4:
        # Return malformed JSON
        return "not a valid json response {{broken"

    else:
        # Success
        return {
            "status": "success",
            "results": [
                {"flight": "IK-101", "price": 450, "departure": "10:00 AM"},
                {"flight": "IK-202", "price": 380, "departure": "2:30 PM"},
            ],
        }

print("Unreliable tool defined (40% failure rate).")
print("Failure modes: TimeoutError (20%) and malformed response (20%).")

Unreliable tool defined (40% failure rate).
Failure modes: TimeoutError (20%) and malformed response (20%).


In [13]:
print("=" * 60)
print("NAIVE AGENT — No error handling")
print("=" * 60)

random.seed(42)

naive_results = []
for i in range(10):
    try:
        result = unreliable_tool("flights to Tokyo")
        if isinstance(result, str):
            parsed = json.loads(result)  # will fail on malformed JSON
        else:
            parsed = result
        naive_results.append({"attempt": i+1, "status": "success", "data": str(parsed)[:60]})
    except Exception as e:
        naive_results.append({"attempt": i+1, "status": "FAILED", "data": str(e)[:60]})

naive_df = pd.DataFrame(naive_results)
print(naive_df.to_string(index=False))

success_rate = (naive_df["status"] == "success").mean()
print(f"\nNaive agent success rate: {success_rate:.0%}")

NAIVE AGENT — No error handling
 attempt  status                                                         data
       1 success {'status': 'success', 'results': [{'flight': 'IK-101', 'pric
       2  FAILED          Tool timed out after 3s for query: flights to Tokyo
       3  FAILED                    Expecting value: line 1 column 1 (char 0)
       4  FAILED                    Expecting value: line 1 column 1 (char 0)
       5 success {'status': 'success', 'results': [{'flight': 'IK-101', 'pric
       6 success {'status': 'success', 'results': [{'flight': 'IK-101', 'pric
       7 success {'status': 'success', 'results': [{'flight': 'IK-101', 'pric
       8  FAILED          Tool timed out after 3s for query: flights to Tokyo
       9 success {'status': 'success', 'results': [{'flight': 'IK-101', 'pric
      10  FAILED          Tool timed out after 3s for query: flights to Tokyo

Naive agent success rate: 50%


In [14]:
def robust_tool_call(query: str, max_retries=3, timeout_s=5) -> dict:
    """Call the unreliable tool with retry logic, timeout, and schema validation."""
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            result = unreliable_tool(query)

            # Schema validation: must be a dict with 'status' and 'results'
            if not isinstance(result, dict):
                raise ValueError(f"Expected dict, got {type(result).__name__}")
            if "status" not in result or "results" not in result:
                raise ValueError(f"Missing required keys: {list(result.keys())}")
            if not isinstance(result["results"], list):
                raise ValueError(f"'results' must be a list")

            return {"ok": True, "data": result, "attempts": attempt}

        except Exception as e:
            last_error = e
            backoff = 0.5 * (2 ** (attempt - 1))  # exponential backoff
            time.sleep(backoff)

    return {"ok": False, "error": str(last_error), "attempts": max_retries}


print("Robust tool caller defined:")
print("  - Max 3 retries")
print("  - Exponential backoff (0.5s, 1s, 2s)")
print("  - Schema validation after every successful call")

Robust tool caller defined:
  - Max 3 retries
  - Exponential backoff (0.5s, 1s, 2s)
  - Schema validation after every successful call


In [15]:
print("=" * 60)
print("ROBUST AGENT — With retries + validation")
print("=" * 60)

random.seed(42)

robust_results = []
for i in range(10):
    result = robust_tool_call("flights to Tokyo")
    robust_results.append({
        "attempt": i+1,
        "status": "success" if result["ok"] else "FAILED",
        "tries": result["attempts"],
        "data": str(result.get("data", result.get("error", "")))[:50],
    })

robust_df = pd.DataFrame(robust_results)
print(robust_df.to_string(index=False))

robust_success = (robust_df["status"] == "success").mean()
print(f"\nRobust agent success rate: {robust_success:.0%}")
print(f"Naive agent success rate:  {success_rate:.0%}")
print(f"\n>>> Improvement: {robust_success - success_rate:+.0%} success rate with retries + validation.")
print(">>> LESSON: Agents are distributed systems. Treat every tool call like a network request.")

ROBUST AGENT — With retries + validation
 attempt  status  tries                                               data
       1 success      1 {'status': 'success', 'results': [{'flight': 'IK-1
       2  FAILED      3                             Expected dict, got str
       3 success      1 {'status': 'success', 'results': [{'flight': 'IK-1
       4 success      1 {'status': 'success', 'results': [{'flight': 'IK-1
       5 success      1 {'status': 'success', 'results': [{'flight': 'IK-1
       6 success      2 {'status': 'success', 'results': [{'flight': 'IK-1
       7 success      3 {'status': 'success', 'results': [{'flight': 'IK-1
       8 success      3 {'status': 'success', 'results': [{'flight': 'IK-1
       9 success      1 {'status': 'success', 'results': [{'flight': 'IK-1
      10 success      2 {'status': 'success', 'results': [{'flight': 'IK-1

Robust agent success rate: 90%
Naive agent success rate:  50%

>>> Improvement: +40% success rate with retries + validation.
>>> LESS

---
## Demo 5: Cost & Latency Explosion in Multi-Step Chains

### The Concept

A single LLM call is cheap and fast. But when agents use **multi-step chains** — calling the LLM multiple times per user query — cost and latency multiply. Without tracking, this becomes a runaway cost problem.

### What We'll Do

1. **Single-shot approach:** Answer a complex question in one LLM call
2. **Agent-loop approach:** Break the same question into sub-tasks, call the LLM for each
3. Compare: tokens used, latency, and estimated cost

### Ops Takeaway

> Cost and latency are **first-class SLOs**. Instrument them from day one.

In [8]:
COMPLEX_QUERY = (
    ''' Find the best round-trip flights from San Francisco to Tokyo within a $800 budget.
    Recommend a 3-night hotel in central Tokyo under $150/night.
    Plan activities for Day 1 in Tokyo (temples and traditional culture). Budget: $100.
    Plan activities for Day 2 in Tokyo (modern Tokyo: Akihabara, Shibuya, Harajuku). Budget: $100.
    Plan activities for Day 3 in Tokyo (food tour and departure). Budget: $80.
    Create a final cost breakdown table for the entire trip. '''
)

# Pricing (approximate, as of early 2026 for gpt-4o-mini)
INPUT_COST_PER_1K  = 0.00015   # $/1K input tokens
OUTPUT_COST_PER_1K = 0.0006    # $/1K output tokens

def get_usage(resp):
    """Extract token usage from a LangChain AIMessage's response_metadata."""
    usage = resp.response_metadata.get("token_usage", {})
    return {
        "prompt_tokens": usage.get("prompt_tokens", 0),
        "completion_tokens": usage.get("completion_tokens", 0),
        "total_tokens": usage.get("total_tokens", 0),
    }

def estimate_cost(usage):
    """Estimate cost from a usage dict with prompt_tokens and completion_tokens."""
    inp = usage["prompt_tokens"]
    out = usage["completion_tokens"]
    return (inp / 1000) * INPUT_COST_PER_1K + (out / 1000) * OUTPUT_COST_PER_1K

print(f"Complex query: \"{COMPLEX_QUERY}\"")
print(f"\nPricing assumptions ({MODEL}):")
print(f"  Input:  ${INPUT_COST_PER_1K}/1K tokens")
print(f"  Output: ${OUTPUT_COST_PER_1K}/1K tokens")

Complex query: " Find the best round-trip flights from San Francisco to Tokyo within a $800 budget.
    Recommend a 3-night hotel in central Tokyo under $150/night.
    Plan activities for Day 1 in Tokyo (temples and traditional culture). Budget: $100.
    Plan activities for Day 2 in Tokyo (modern Tokyo: Akihabara, Shibuya, Harajuku). Budget: $100.
    Plan activities for Day 3 in Tokyo (food tour and departure). Budget: $80.
    Create a final cost breakdown table for the entire trip. "

Pricing assumptions (gpt-4o-mini):
  Input:  $0.00015/1K tokens
  Output: $0.0006/1K tokens


In [9]:
print("=" * 60)
print("APPROACH 1: Single-Shot (one LLM call)")
print("=" * 60)

resp_single, single_latency = chat(
    COMPLEX_QUERY,
    system="You are a travel planning assistant. Be thorough but concise.",
    temperature=0,
    max_tokens=800,
)
single_usage = get_usage(resp_single)
single_cost = estimate_cost(single_usage)

print(f"\nResponse preview: {resp_single.content[:200]}...")
print(f"\nTokens — input: {single_usage['prompt_tokens']}, output: {single_usage['completion_tokens']}, total: {single_usage['total_tokens']}")
print(f"Latency: {single_latency:.2f}s")
print(f"Estimated cost: ${single_cost:.6f}")

APPROACH 1: Single-Shot (one LLM call)

Response preview: ### Flight Options
**Round-Trip Flights from San Francisco (SFO) to Tokyo (NRT/HND)**  
- **Airline:** United Airlines  
- **Dates:** Departing on March 1, 2024, and returning on March 4, 2024  
- **P...

Tokens — input: 139, output: 800, total: 939
Latency: 9.40s
Estimated cost: $0.000501


In [10]:
print("=" * 60)
print("APPROACH 2: Agent Loop (multiple LLM calls)")
print("=" * 60)

SUB_TASKS = [
    "Find the best round-trip flights from San Francisco to Tokyo within a $800 budget.",
    "Recommend a 3-night hotel in central Tokyo under $150/night.",
    "Plan activities for Day 1 in Tokyo (temples and traditional culture). Budget: $100.",
    "Plan activities for Day 2 in Tokyo (modern Tokyo: Akihabara, Shibuya, Harajuku). Budget: $100.",
    "Plan activities for Day 3 in Tokyo (food tour and departure). Budget: $80.",
    "Create a final cost breakdown table for the entire trip.",
]

total_input_tokens = 0
total_output_tokens = 0
total_latency = 0
total_cost = 0

step_data = []

for i, task in enumerate(SUB_TASKS):
    resp, latency = chat(
        task,
        system="You are a travel planning assistant. Be concise.",
        temperature=0,
        max_tokens=400,
    )
    usage = get_usage(resp)
    cost = estimate_cost(usage)

    total_input_tokens += usage["prompt_tokens"]
    total_output_tokens += usage["completion_tokens"]
    total_latency += latency
    total_cost += cost

    step_data.append({
        "step": i + 1,
        "task": task[:50] + "...",
        "input_tok": usage["prompt_tokens"],
        "output_tok": usage["completion_tokens"],
        "latency_s": round(latency, 2),
        "cost_usd": round(cost, 6),
    })
    print(f"  Step {i+1}: {latency:.2f}s | {usage['total_tokens']} tokens | ${cost:.6f}")

agent_df = pd.DataFrame(step_data)
print(f"\nTotal — input: {total_input_tokens}, output: {total_output_tokens}")
print(f"Total latency: {total_latency:.2f}s")
print(f"Total cost:    ${total_cost:.6f}")

APPROACH 2: Agent Loop (multiple LLM calls)
  Step 1: 2.14s | 232 tokens | $0.000122
  Step 2: 1.35s | 108 tokens | $0.000049
  Step 3: 2.63s | 318 tokens | $0.000172
  Step 4: 2.44s | 301 tokens | $0.000158
  Step 5: 2.72s | 239 tokens | $0.000125
  Step 6: 1.17s | 127 tokens | $0.000062

Total — input: 236, output: 1089
Total latency: 12.46s
Total cost:    $0.000689


In [11]:
print("\n" + "=" * 60)
print("COMPARISON: Single-Shot vs Agent Loop")
print("=" * 60)

comparison = pd.DataFrame([
    {
        "Approach": "Single-Shot",
        "LLM Calls": 1,
        "Total Tokens": single_usage["total_tokens"],
        "Latency (s)": round(single_latency, 2),
        "Cost ($)": round(single_cost, 6),
    },
    {
        "Approach": "Agent Loop",
        "LLM Calls": len(SUB_TASKS),
        "Total Tokens": total_input_tokens + total_output_tokens,
        "Latency (s)": round(total_latency, 2),
        "Cost ($)": round(total_cost, 6),
    },
])

print(comparison.to_string(index=False))

token_multiplier = (total_input_tokens + total_output_tokens) / single_usage["total_tokens"]
latency_multiplier = total_latency / single_latency
cost_multiplier = total_cost / single_cost

print(f"\nAgent loop used:")
print(f"  {token_multiplier:.1f}x more tokens")
print(f"  {latency_multiplier:.1f}x more latency")
print(f"  {cost_multiplier:.1f}x more cost")
print(f"\n>>> At 10,000 queries/day, the agent loop costs ${total_cost * 10000:.2f}/day")
print(f">>> vs single-shot at ${single_cost * 10000:.2f}/day")
print(f"\n>>> LESSON: Cost and latency are FIRST-CLASS SLOs.")
print(f">>> Every additional LLM call in a chain multiplies both.")
print(f">>> Instrument cost-per-request from day one.")


COMPARISON: Single-Shot vs Agent Loop
   Approach  LLM Calls  Total Tokens  Latency (s)  Cost ($)
Single-Shot          1           939         9.40  0.000501
 Agent Loop          6          1325        12.46  0.000689

Agent loop used:
  1.4x more tokens
  1.3x more latency
  1.4x more cost

>>> At 10,000 queries/day, the agent loop costs $6.89/day
>>> vs single-shot at $5.01/day

>>> LESSON: Cost and latency are FIRST-CLASS SLOs.
>>> Every additional LLM call in a chain multiplies both.
>>> Instrument cost-per-request from day one.


---
## Summary: Why LLMOps?

You've now seen five concrete failure modes that **do not exist in traditional software**:

| Demo | Failure Mode | Traditional Software | LLM Systems |
|---|---|---|---|
| 1 | Output variance | Same input → same output | Same input → different output |
| 2 | Prompt drift | Code changes are tested | Prompt changes often skip testing |
| 3 | Hallucination | App either works or errors | Model confidently returns wrong answers |
| 4 | Tool failure in agents | API errors are caught | Tool failures crash multi-step chains |
| 5 | Cost/latency explosion | Compute is predictable | Token costs multiply with each chain step |
